# Experiment 30 - High Signal Feature Attack

**Target: 0.961+ OOF ROC-AUC**

This experiment changes the feature representation rather than doing another small XGBoost hyperparameter sweep.

It tests:
- exact-value target encoding
- exact-value frequency encoding
- binned numerical representations
- pair and triple group encodings
- derived ratio features
- several XGBoost configurations

Validation uses nested leakage-safe encoding.

This experiment intentionally does **not** generate a submission CSV or OOF prediction CSV.


In [1]:

import os
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 42
N_OUTER = 3
N_INNER = 3

PROJECT_DIR = r"C:\Users\aakif\Documents\DataCompetition"

TRAIN_PATH = os.path.join(PROJECT_DIR, "data", "train.csv")

# Fallback in case the dataset is stored directly in the project root.
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = os.path.join(PROJECT_DIR, "train.csv")

train = pd.read_csv(TRAIN_PATH)

TARGET = "Will_Buy_EV"
ID_COL = "id"

if TARGET not in train.columns:
    raise ValueError(f"Target column '{TARGET}' was not found.")

if ID_COL not in train.columns:
    raise ValueError(f"ID column '{ID_COL}' was not found.")

print("Train shape:", train.shape)
print("Target values:", train[TARGET].value_counts(dropna=False).to_dict())


Train shape: (668665, 15)
Target values: {'No': 551886, 'Yes': 116779}


In [2]:

# Convert the competition's Yes/No target safely.
target_values = train[TARGET].astype(str).str.strip()

target_map = {
    "No": 0,
    "Yes": 1,
}

unknown_targets = sorted(set(target_values.unique()) - set(target_map))

if unknown_targets:
    raise ValueError(
        f"Unexpected target values found: {unknown_targets}. "
        f"Expected only {list(target_map.keys())}."
    )

y = target_values.map(target_map).astype(np.int8)

X_raw = train.drop(columns=[TARGET, ID_COL]).copy()

print("Feature shape:", X_raw.shape)
print("Positive rate:", round(float(y.mean()), 6))


Feature shape: (668665, 13)
Positive rate: 0.174645


In [3]:

# Columns used to create high-signal group representations.
#
# These are intentionally based on exact observed values rather than
# broad categorical assumptions.

IDENTITY_COLS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

PAIR_COLS = [
    ("Age", "Annual_Income_USD"),
    ("Age", "Daily_Commute_km"),
    ("Age", "Current_Car_Type"),
    ("Age", "City_Type"),
    ("Annual_Income_USD", "Current_Car_Type"),
    ("Annual_Income_USD", "City_Type"),
    ("Annual_Income_USD", "Range_Anxiety_Level"),
    ("Daily_Commute_km", "Current_Car_Type"),
    ("Daily_Commute_km", "City_Type"),
    ("Daily_Commute_km", "Range_Anxiety_Level"),
    ("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"),
    ("Environmental_Concern_Level", "Range_Anxiety_Level"),
    ("Environmental_Concern_Level", "Current_Car_Type"),
    ("Environmental_Concern_Level", "City_Type"),
    ("Charging_Stations_Near_Home", "Current_Car_Type"),
    ("Charging_Stations_Near_Work", "Current_Car_Type"),
]

TRIPLE_COLS = [
    ("Age", "Current_Car_Type", "City_Type"),
    ("Age", "Annual_Income_USD", "Current_Car_Type"),
    ("Age", "Daily_Commute_km", "Current_Car_Type"),
    ("Annual_Income_USD", "Current_Car_Type", "City_Type"),
    ("Annual_Income_USD", "Range_Anxiety_Level", "Current_Car_Type"),
    ("Daily_Commute_km", "Current_Car_Type", "City_Type"),
    ("Environmental_Concern_Level", "Range_Anxiety_Level", "Current_Car_Type"),
    ("Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Current_Car_Type"),
]

BIN_GROUPS = [
    ("Age_BIN5",),
    ("Age_BIN10",),
    ("Annual_Income_USD_BIN10",),
    ("Daily_Commute_km_BIN5",),
    ("Daily_Commute_km_BIN10",),
    ("Charging_Stations_Near_Home_BIN5",),
    ("Charging_Stations_Near_Work_BIN5",),
    ("Age_BIN5", "Current_Car_Type"),
    ("Age_BIN10", "City_Type"),
    ("Annual_Income_USD_BIN10", "Current_Car_Type"),
    ("Daily_Commute_km_BIN5", "Current_Car_Type"),
    ("Daily_Commute_km_BIN10", "Range_Anxiety_Level"),
]

required_columns = set(IDENTITY_COLS)

for group in PAIR_COLS + TRIPLE_COLS:
    required_columns.update(group)

for group in BIN_GROUPS:
    required_columns.update(group)

missing = sorted(required_columns - set(X_raw.columns))

# BIN_GROUPS contain derived columns, so only check the original columns here.
derived_bin_columns = {
    "Age_BIN5",
    "Age_BIN10",
    "Annual_Income_USD_BIN10",
    "Daily_Commute_km_BIN5",
    "Daily_Commute_km_BIN10",
    "Charging_Stations_Near_Home_BIN5",
    "Charging_Stations_Near_Work_BIN5",
}

missing_original = sorted(
    (required_columns - derived_bin_columns) - set(X_raw.columns)
)

if missing_original:
    raise ValueError(f"Required source columns are missing: {missing_original}")

print("Feature-group definitions loaded.")


Feature-group definitions loaded.


In [4]:

def add_derived_features(df):
    """
    Add deterministic features that do not use the target.
    """
    out = df.copy()

    numeric_bins = {
        "Age": [5, 10],
        "Annual_Income_USD": [5, 10],
        "Daily_Commute_km": [5, 10],
        "Charging_Stations_Near_Home": [5, 10],
        "Charging_Stations_Near_Work": [5, 10],
    }

    for col, widths in numeric_bins.items():
        if col not in out.columns:
            continue

        values = pd.to_numeric(out[col], errors="coerce")

        for width in widths:
            out[f"{col}_BIN{width}"] = (
                np.floor(values / width) * width
            ).astype("float64")

    # Ratio features are target-free and therefore safe to calculate globally.
    income = pd.to_numeric(out["Annual_Income_USD"], errors="coerce")
    age = pd.to_numeric(out["Age"], errors="coerce")
    commute = pd.to_numeric(out["Daily_Commute_km"], errors="coerce")
    home_station = pd.to_numeric(
        out["Charging_Stations_Near_Home"], errors="coerce"
    )
    work_station = pd.to_numeric(
        out["Charging_Stations_Near_Work"], errors="coerce"
    )

    out["Income_Per_Age"] = income / age.replace(0, np.nan)
    out["Commute_Per_Home_Station"] = (
        commute / home_station.replace(0, np.nan)
    )
    out["Commute_Per_Work_Station"] = (
        commute / work_station.replace(0, np.nan)
    )

    out = out.replace([np.inf, -np.inf], np.nan)

    return out


X_engineered = add_derived_features(X_raw)

print("Engineered feature shape:", X_engineered.shape)


Engineered feature shape: (668665, 26)


In [5]:

def make_group_key(df, columns):
    """
    Create a deterministic composite key for one or more columns.

    String conversion makes the function work consistently with mixed
    numeric and categorical columns. Missing values receive a dedicated
    token.
    """
    parts = []

    for col in columns:
        values = df[col].astype("string").fillna("__NA__")
        parts.append(values)

    key = parts[0].astype(str)

    for part in parts[1:]:
        key = key + "||" + part.astype(str)

    return key


def fit_group_statistics(df, target, groups):
    """
    Fit target-encoding and frequency statistics using ONLY df and target.
    """
    global_mean = float(target.mean())

    stats = {}

    for columns in groups:
        key = make_group_key(df, columns)

        temp = pd.DataFrame({
            "__key__": key.to_numpy(),
            "__target__": np.asarray(target),
        })

        grouped = temp.groupby("__key__", sort=False)["__target__"].agg(
            ["mean", "count"]
        )

        frequency = key.value_counts(dropna=False)

        stats[tuple(columns)] = {
            "target_mean": grouped["mean"],
            "frequency": frequency,
        }

    return stats, global_mean


def transform_group_statistics(df, stats, global_mean):
    """
    Apply already-fitted target/frequency statistics to df.

    Unseen groups receive the fitting-set global target mean and zero
    frequency. No information from df itself is used.
    """
    features = {}

    for columns, group_stats in stats.items():
        key = make_group_key(df, columns)

        target_name = "TE_" + "__".join(columns)
        freq_name = "FE_" + "__".join(columns)

        target_values = key.map(group_stats["target_mean"])
        frequency_values = key.map(group_stats["frequency"])

        features[target_name] = (
            target_values.fillna(global_mean).astype("float32")
        )

        features[freq_name] = (
            frequency_values.fillna(0).astype("float32")
        )

    return pd.DataFrame(features, index=df.index)


In [6]:

def prepare_base_features(train_df, valid_df):
    """
    One-hot encode ordinary categorical columns using the outer-training
    data only, then align validation columns.
    """
    train_part = train_df.copy()
    valid_part = valid_df.copy()

    categorical_cols = train_part.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()

    train_part = pd.get_dummies(
        train_part,
        columns=categorical_cols,
        dummy_na=True,
    )

    valid_part = pd.get_dummies(
        valid_part,
        columns=categorical_cols,
        dummy_na=True,
    )

    valid_part = valid_part.reindex(
        columns=train_part.columns,
        fill_value=0,
    )

    # Ensure numeric matrix.
    train_part = train_part.apply(pd.to_numeric, errors="coerce")
    valid_part = valid_part.apply(pd.to_numeric, errors="coerce")

    train_part = train_part.replace([np.inf, -np.inf], np.nan)
    valid_part = valid_part.replace([np.inf, -np.inf], np.nan)

    train_part = train_part.fillna(0)
    valid_part = valid_part.fillna(0)

    return train_part.astype("float32"), valid_part.astype("float32")


In [7]:

def build_nested_outer_features(
    outer_train_raw,
    outer_train_y,
    outer_valid_raw,
    identity_cols,
    pair_cols,
    triple_cols,
    bin_groups,
    seed=SEED,
):
    """
    Build leakage-safe features for ONE outer CV fold.

    Training rows:
        target/frequency encodings are generated through inner OOF folds.

    Validation rows:
        statistics are fitted on the complete outer-training fold only.

    This prevents target information from leaking into either:
        1. the model's training rows
        2. the outer validation rows
    """
    all_groups = []

    for col in identity_cols:
        all_groups.append((col,))

    all_groups.extend(pair_cols)
    all_groups.extend(triple_cols)
    all_groups.extend(bin_groups)

    inner_cv = StratifiedKFold(
        n_splits=N_INNER,
        shuffle=True,
        random_state=seed,
    )

    train_extra = pd.DataFrame(
        index=outer_train_raw.index,
        dtype="float32",
    )

    # True OOF encoding for the outer-training rows.
    for inner_train_idx, inner_valid_idx in inner_cv.split(
        outer_train_raw,
        outer_train_y,
    ):
        inner_train = outer_train_raw.iloc[inner_train_idx]
        inner_valid = outer_train_raw.iloc[inner_valid_idx]

        inner_y_train = outer_train_y.iloc[inner_train_idx]

        fitted_stats, inner_global_mean = fit_group_statistics(
            inner_train,
            inner_y_train,
            all_groups,
        )

        inner_features = transform_group_statistics(
            inner_valid,
            fitted_stats,
            inner_global_mean,
        )

        train_extra.loc[
            inner_valid.index,
            inner_features.columns,
        ] = inner_features.astype("float32")

    # Reorder explicitly to the original outer-training row order.
    train_extra = train_extra.reindex(outer_train_raw.index)

    if train_extra.isna().any().any():
        raise RuntimeError(
            "Nested OOF encoding produced missing training values. "
            "Check the inner CV construction."
        )

    # Validation statistics are fitted ONLY on the outer-training fold.
    outer_stats, outer_global_mean = fit_group_statistics(
        outer_train_raw,
        outer_train_y,
        all_groups,
    )

    valid_extra = transform_group_statistics(
        outer_valid_raw,
        outer_stats,
        outer_global_mean,
    )

    train_extra = train_extra.reset_index(drop=True)
    valid_extra = valid_extra.reset_index(drop=True)

    # Base features are also fit/aligned using only outer-training data.
    train_base, valid_base = prepare_base_features(
        outer_train_raw,
        outer_valid_raw,
    )

    train_base = train_base.reset_index(drop=True)
    valid_base = valid_base.reset_index(drop=True)

    train_features = pd.concat(
        [train_base, train_extra],
        axis=1,
    )

    valid_features = pd.concat(
        [valid_base, valid_extra],
        axis=1,
    )

    valid_features = valid_features.reindex(
        columns=train_features.columns,
        fill_value=0,
    )

    train_features = train_features.astype("float32")
    valid_features = valid_features.astype("float32")

    return train_features, valid_features


In [8]:

# XGBoost configurations.
#
# These are intentionally meaningfully different from the tiny refinement
# sweep in Experiment 29.

MODEL_CONFIGS = {
    "XGB_30_A_balanced": {
        "n_estimators": 1200,
        "max_depth": 5,
        "learning_rate": 0.025,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
    },
    "XGB_30_B_interaction": {
        "n_estimators": 1000,
        "max_depth": 6,
        "learning_rate": 0.030,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.95,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
    },
    "XGB_30_C_regularized": {
        "n_estimators": 1200,
        "max_depth": 5,
        "learning_rate": 0.025,
        "min_child_weight": 4,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.10,
        "reg_lambda": 2.0,
    },
    "XGB_30_D_deeper": {
        "n_estimators": 1000,
        "max_depth": 7,
        "learning_rate": 0.025,
        "min_child_weight": 3,
        "subsample": 0.85,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
    },
}

print("Models:", list(MODEL_CONFIGS))


Models: ['XGB_30_A_balanced', 'XGB_30_B_interaction', 'XGB_30_C_regularized', 'XGB_30_D_deeper']


In [9]:

def make_model(params):
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
        **params,
    )


In [10]:

# Prepare all engineered features before CV.
#
# These features are target-free. The target-dependent encodings are built
# separately inside each outer fold.

X_cv = X_engineered.copy()

# Make sure every declared group column exists after feature engineering.
required_groups = []

for col in IDENTITY_COLS:
    required_groups.append((col,))

required_groups.extend(PAIR_COLS)
required_groups.extend(TRIPLE_COLS)
required_groups.extend(BIN_GROUPS)

missing_group_columns = sorted(
    {
        col
        for group in required_groups
        for col in group
        if col not in X_cv.columns
    }
)

if missing_group_columns:
    raise ValueError(
        f"Group columns missing after feature engineering: "
        f"{missing_group_columns}"
    )

print("CV feature matrix:", X_cv.shape)


CV feature matrix: (668665, 26)


In [11]:

outer_cv = StratifiedKFold(
    n_splits=N_OUTER,
    shuffle=True,
    random_state=SEED,
)

fold_indices = list(outer_cv.split(X_cv, y))

results = []

# IMPORTANT:
# The expensive nested target/frequency encoding is now built ONCE per
# outer fold and reused by every model.
#
# Previously, the encoding was rebuilt for every model, causing:
#     4 models x 3 folds = 12 expensive encoding builds
#
# Now:
#     3 folds = 3 encoding builds
#
# The 4 models still train independently on the exact same leakage-safe
# features for each fold.

for fold_number, (outer_train_idx, outer_valid_idx) in enumerate(
    fold_indices,
    start=1,
):
    print("\n" + "=" * 80)
    print(f"Preparing outer fold {fold_number}/{N_OUTER}")
    print("=" * 80)

    outer_train_raw = X_cv.iloc[outer_train_idx].copy()
    outer_valid_raw = X_cv.iloc[outer_valid_idx].copy()

    outer_train_y = y.iloc[outer_train_idx].copy()
    outer_valid_y = y.iloc[outer_valid_idx].copy()

    print(
        f"Train rows: {len(outer_train_idx):,} | "
        f"Validation rows: {len(outer_valid_idx):,}"
    )

    # Build the expensive leakage-safe feature representation ONCE.
    train_features, valid_features = build_nested_outer_features(
        outer_train_raw=outer_train_raw,
        outer_train_y=outer_train_y,
        outer_valid_raw=outer_valid_raw,
        identity_cols=IDENTITY_COLS,
        pair_cols=PAIR_COLS,
        triple_cols=TRIPLE_COLS,
        bin_groups=BIN_GROUPS,
        seed=SEED + fold_number,
    )

    print(
        f"Feature matrix: {train_features.shape[0]:,} x "
        f"{train_features.shape[1]:,}"
    )

    # Train all models using the same already-built leakage-safe features.
    for model_name, params in MODEL_CONFIGS.items():
        print("\n" + "-" * 70)
        print(f"{model_name} | Fold {fold_number}/{N_OUTER}")
        print("-" * 70)

        model = make_model(params)

        model.fit(
            train_features,
            outer_train_y,
        )

        valid_pred = model.predict_proba(valid_features)[:, 1]

        fold_auc = roc_auc_score(
            outer_valid_y,
            valid_pred,
        )

        results.append({
            "model": model_name,
            "fold": fold_number,
            "roc_auc": float(fold_auc),
        })

        print(f"ROC-AUC: {fold_auc:.6f}")

        del model
        del valid_pred

    # Free the large fold-specific matrices before moving to the next fold.
    del (
        outer_train_raw,
        outer_valid_raw,
        outer_train_y,
        outer_valid_y,
        train_features,
        valid_features,
    )


# Convert fold-level results into the final model ranking.
fold_results_df = pd.DataFrame(results)

results_df = (
    fold_results_df
    .groupby("model", as_index=False)
    .agg(
        oof_roc_auc=("roc_auc", "mean"),
        fold_std=("roc_auc", "std"),
    )
)

fold_table = (
    fold_results_df
    .pivot(
        index="model",
        columns="fold",
        values="roc_auc",
    )
    .rename(columns={
        1: "fold_1",
        2: "fold_2",
        3: "fold_3",
    })
    .reset_index()
)

results_df = (
    results_df
    .merge(fold_table, on="model", how="left")
    .sort_values("oof_roc_auc", ascending=False)
    .reset_index(drop=True)
)

# std() returns NaN only if there is one fold, which cannot happen here,
# but keep the result clean just in case.
results_df["fold_std"] = results_df["fold_std"].fillna(0.0)

results_df



Preparing outer fold 1/3
Train rows: 445,776 | Validation rows: 222,889
Feature matrix: 445,776 x 129

----------------------------------------------------------------------
XGB_30_A_balanced | Fold 1/3
----------------------------------------------------------------------
ROC-AUC: 0.944120

----------------------------------------------------------------------
XGB_30_B_interaction | Fold 1/3
----------------------------------------------------------------------
ROC-AUC: 0.943384

----------------------------------------------------------------------
XGB_30_C_regularized | Fold 1/3
----------------------------------------------------------------------
ROC-AUC: 0.944222

----------------------------------------------------------------------
XGB_30_D_deeper | Fold 1/3
----------------------------------------------------------------------
ROC-AUC: 0.943596

Preparing outer fold 2/3
Train rows: 445,777 | Validation rows: 222,888
Feature matrix: 445,777 x 129

-----------------------------

,model,oof_roc_auc,fold_std,fold_1,fold_2,fold_3
0,XGB_30_C_regularized,0.944820,0.000551,0.944222,0.945307,0.944931
1,XGB_30_A_balanced,0.944614,0.000435,0.944120,0.944939,0.944783
2,XGB_30_D_deeper,0.944104,0.000481,0.943596,0.944552,0.944165
3,XGB_30_B_interaction,0.944008,0.000549,0.943384,0.944415,0.944226


In [12]:

# Save ONLY the compact model-results table.
#
# No submission predictions and no OOF prediction arrays are written.

RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "experiment_30_model_results.csv",
)

results_df.to_csv(RESULTS_PATH, index=False)

best_score = float(results_df.iloc[0]["oof_roc_auc"])
best_model = str(results_df.iloc[0]["model"])

print("\nExperiment 30 results")
print(results_df.to_string(index=False))

print(f"\nBest model: {best_model}")
print(f"Best OOF ROC-AUC: {best_score:.6f}")
print(f"Target 0.961+: {'REACHED' if best_score >= 0.961 else 'NOT REACHED'}")
print(f"\nSaved results only to: {RESULTS_PATH}")



Experiment 30 results
               model  oof_roc_auc  fold_std   fold_1   fold_2   fold_3
XGB_30_C_regularized     0.944820  0.000551 0.944222 0.945307 0.944931
   XGB_30_A_balanced     0.944614  0.000435 0.944120 0.944939 0.944783
     XGB_30_D_deeper     0.944104  0.000481 0.943596 0.944552 0.944165
XGB_30_B_interaction     0.944008  0.000549 0.943384 0.944415 0.944226

Best model: XGB_30_C_regularized
Best OOF ROC-AUC: 0.944820
Target 0.961+: NOT REACHED

Saved results only to: C:\Users\aakif\Documents\DataCompetition\results\experiment_30_model_results.csv
